In [2]:
import gym
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback, ProgressBarCallback, CallbackList
import matplotlib.pyplot as plt
import numpy as np


2025-01-05 10:00:04.067371: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-05 10:00:04.256630: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736064004.320203   40962 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736064004.337434   40962 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-05 10:00:04.494924: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
# наша модель має приймати калбеки, тож ми створимо калбек, щоб дивитись на наші успіхи в процесі навчання
class RenderCallback(BaseCallback):
    def __init__(self, env, visualize_every_n_steps=1000):
        super(RenderCallback, self).__init__()
        self.env = env
        self.visualize_every_n_steps = visualize_every_n_steps
        self.steps = 0

    def _on_step(self) -> bool:
        self.steps += 1
        if self.steps % self.visualize_every_n_steps == 0:  # рендерінг віртуального середовища може уповілнювати процес навчання, тож будемо дивитись на прогресс за кожні n кроків
            print(f"Visualizing step {self.steps}")
            for i in range(900):
                self.env.render()

        # print(f"Step {self.steps}, Reward: {self.locals['rewards']}")

        return True

In [3]:
1e-4

0.0001

In [3]:
# завантажуємо модель і середовищє
env_name = "CarRacing-v2"
# env_name = "MountainCar-v0"
env = gym.make(env_name, render_mode=None)
model = PPO("CnnPolicy", env, verbose=0, learning_rate=1e-4, ent_coef=0.05, batch_size=128)

/home/pivden/hilel_machine_learning/hilel_machine_learning/lib/python3.10/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [6]:
class RewardPlotCallback(BaseCallback):
    def __init__(self, output_file="rewards.png", freq=10):
        super(RewardPlotCallback, self).__init__()
        self.episode_rewards = []  # Список для збереження очок кожного епізоду
        self.output_file = output_file  # Шлях до файлу для збереження графіка
        self.current_episode_reward = 0  # Поточна винагорода за епізод

    def _on_step(self) -> bool:
        # Додаємо винагороду за поточний крок до поточної епізодної винагороди
        self.current_episode_reward += self.locals["rewards"][0]

        # Якщо епізод завершено
        if self.locals["dones"][0]:
            # Додаємо повну винагороду за епізод до списку
            self.episode_rewards.append(self.current_episode_reward)
            self.current_episode_reward = 0  # Скидаємо поточну винагороду
            # Оновлюємо графік
            self.plot_rewards()
        return True

    def plot_rewards(self):
        # Побудова графіка
        plt.figure(figsize=(15, 6))
        plt.plot(self.episode_rewards, label="reward per episode", color="blue")

        # Додавання лінії тренду
        if len(self.episode_rewards) > 1:
            x = np.arange(len(self.episode_rewards))
            z = np.polyfit(x, self.episode_rewards, 1)  # Лінія тренду першого ступеня
            p = np.poly1d(z)
            plt.plot(x, p(x), label="trend", color="red", linestyle="--")

        # Оформлення графіка
        plt.xlabel("episode")
        plt.ylabel("reward")
        plt.legend()
        plt.grid()

        # Зберігаємо графік у файл
        plt.savefig(self.output_file)
        plt.close()


In [7]:
eval_env = gym.make(env_name, render_mode=None)

eval_callback = EvalCallback(eval_env, best_model_save_path="./logs/",
                             log_path="./logs/", eval_freq=10000,
                             deterministic=True, render=False, n_eval_episodes=10)


plot_callback = RewardPlotCallback()

In [7]:
callbacks = CallbackList([plot_callback, eval_callback])

In [ ]:
# callback = RenderCallback(env, visualize_every_n_steps=1000)  #з рендером награлись доволі швидко - він уповільнює нас в 4 рази

timesteps = 3000000
model.learn(total_timesteps=timesteps, progress_bar=True, callback=eval_callback)

Output()

/home/pivden/hilel_machine_learning/hilel_machine_learning/lib/python3.10/site-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7e9969099060> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7e996535bf10>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


/home/pivden/hilel_machine_learning/hilel_machine_learning/lib/python3.10/site-packages/gym/utils/passive_env_check
er.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):

/home/pivden/hilel_machine_learning/hilel_machine_learning/lib/python3.10/site-packages/stable_baselines3/common/ev
aluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in 
reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping 
environment first with ``Monitor`` wrapper.
  warnings.warn(

Eval num_timesteps=10000, episode_reward=-56.19 +/- 12.87

...

Episode length: 1000.00 +/- 0.00

Eval num_timesteps=2380000, episode_reward=327.04 +/- 199.29

Episode length: 1000.00 +/- 0.00

Eval num_timesteps=2390000, episode_reward=288.79 +/- 137.04

Episode length: 1000.00 +/- 0.00

In [ ]:
# на 30% дочитав документацію до політик моделі і стало ясно, що варто було використовувати CnnPolicy для нашого середовища.
# Але вже нехай навчиться. Потім навчимо з CnnPolicy і порівняємо результати


# UPD: не будемо ексепементувати. Наша модель з MlpPolicy мала майже стаціонарний графік, що казало просто про випадкові дії 
# та після 1.1 мільйона кроків поросто зависла на приблизно -93 очок
# CnnPolicy вже на 60000 кроці на валідації дав середню винагороду в 36, а на 80к - 138.16. При цьому наблюдається нестаціонарний графік з ярковираженим трендом

In [34]:
# Збереження моделі

model.save(env_name)
env.close()


In [ ]:
# Завантаження середовища з вказаним render_mode
env = gym.make(env_name, render_mode="human")

# Завантаження навченого агента
# last_model = PPO.load(env_name)
best_model = PPO.load(f"logs/best_model")



/home/pivden/hilel_machine_learning/hilel_machine_learning/lib/python3.10/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [9]:
def eval_agent(env, model):
    obs, _ = env.reset() 
    for _ in range(2000):
        env.render()  # Рендеринг середовища
        action, _states = model.predict(obs)  # Прогноз дій
        
        result = env.step(action)
        obs, reward, terminated, truncated, info = result
        done = terminated or truncated
    
        if done:
            obs, _ = env.reset()

    env.close()

In [10]:
eval_agent(env, best_model)

/home/pivden/hilel_machine_learning/hilel_machine_learning/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):
